# Preprocesamiento, Ventaneo y División del Dataset para MyoTensor Proto (S07)

Este notebook realiza el pipeline completo de procesamiento para el canal único de sEMG de **MyoTensor Proto** (S07):
1. Filtro de outliers (`valid_flag == 1`).
2. Segmentación contigua por bloques de estado de estímulo (evita mezclar tiempos no contiguos).
3. Extracción de ventanas deslizantes con solapamiento ($W=300$ ms, $S=50$ ms).
4. División en conjuntos de **Entrenamiento (80%)** y **Prueba (20%)** estratificados por clase y bloques contiguos.
5. Exportación de tensores en formato NumPy binario (`.npy`), incluyendo etiquetas categóricas (para SVM/RF) y etiquetas one-hot (para tu red neuronal CNN-LSTM).

---

## ¿Qué Estructura Tienen los Tensores Finales?

### Conjunto de Entrenamiento (80%):
*   **`X_train.npy`:** Tensor 3D de shape `(3592, 300, 1)`
*   **`y_train.npy`:** Matriz 2D de shape `(3592, 4)` con codificación one-hot para tu red **CNN-LSTM**.

### Conjunto de Prueba/Validación (20%):
*   **`X_test.npy`:** Tensor 3D de shape `(482, 300, 1)`
*   **`y_test.npy`:** Matriz 2D de shape `(482, 4)` con codificación one-hot para tu red **CNN-LSTM**.

## 1. Configuración de Parámetros Globales

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras.utils import to_categorical

# Función puramente robusta para buscar y cargar el archivo .env
def load_env_variables():
    import os
    from pathlib import Path
    
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

W = 300         # Tamaño de la ventana (300 ms a 1000 Hz)
S = 50          # Paso de la ventana (50 ms para solapamiento continuo)
TEST_SIZE = 0.2 # 20% para el conjunto de prueba (Test/Validation)
SEED = 42       # Semilla para partición reproducible

BASE_DIR = Path(os.environ["RAW_DATA_PROTO"])
OUTPUT_DIR = Path(os.environ["PROCESSED_TENSOR_PROTO"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

2026-05-31 20:53:05.425461: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-31 20:53:05.461539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-31 20:53:06.240828: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## 2. Algoritmo de Preprocesamiento, Partición Estratificada por Bloques y Ventaneo Aislado

In [2]:
X_list = []
y_list = []

# Buscar específicamente la sesión CSV de S07
csv_files = sorted(list(BASE_DIR.glob("S07/session_*.csv")))
print(f"[*] PASO 1: Buscando archivos CSV en {BASE_DIR}")
print(f"    -> Se encontró el archivo de S07: {[f.name for f in csv_files]}\n")

csv_path = csv_files[0]
print(f"[*] PASO 2: Cargando archivo {csv_path.name}...")
df_raw = pd.read_csv(csv_path)
print(f"    -> Muestras totales en crudo: {len(df_raw)}")

# 1. Limpieza inicial: Filtro de muestras de calidad valid_flag == 1
df_clean = df_raw[df_raw["valid_flag"] == 1].copy()
n_outliers = len(df_raw) - len(df_clean)
print(f"[*] PASO 3: Aplicando filtro de calidad (valid_flag == 1)")
print(f"    -> Muestras válidas conservadas: {len(df_clean)} (outliers eliminados: {n_outliers})")

# 2. Partición por Bloques (Split First) estratificada por tipo de estímulo (Activo vs Reposo)
print(f"\n[*] PASO 4: Dividiendo el dataset por bloques de forma estratificada...")

# Filtrar activos y reposos por separado
df_active = df_clean[df_clean["restimulus"] != 0].copy()
df_reposo = df_clean[df_clean["restimulus"] == 0].copy()

# A) Split para Gestos Activos por Repetición
print("   -> Dividiendo Gestos Activos por repetición (group_id = re_repetition_id)...")
gss_active = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx_act, test_idx_act = next(gss_active.split(df_active, groups=df_active["re_repetition_id"]))
df_train_act = df_active.iloc[train_idx_act].copy()
df_test_act = df_active.iloc[test_idx_act].copy()

# B) Split para Reposo por Bloques Contiguos
print("   -> Dividiendo Reposo por bloques contiguos (block_id)...")
reposo_blocks = (df_clean["restimulus"] != df_clean["restimulus"].shift()).cumsum()
df_reposo["block_id"] = reposo_blocks[df_clean["restimulus"] == 0]

gss_reposo = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx_rep, test_idx_rep = next(gss_reposo.split(df_reposo, groups=df_reposo["block_id"]))
df_train_rep = df_reposo.iloc[train_idx_rep].copy()
df_test_rep = df_reposo.iloc[test_idx_rep].copy()

# C) Combinar
df_train = pd.concat([df_train_act, df_train_rep]).sort_index()
df_test = pd.concat([df_test_act, df_test_rep]).sort_index()

# Para compatibilidad con groupby en ventaneo
df_train["group_id"] = np.where(df_train["restimulus"] == 0, df_train["block_id"], df_train["re_repetition_id"])
df_test["group_id"] = np.where(df_test["restimulus"] == 0, df_test["block_id"], df_test["re_repetition_id"])
df_clean["group_id"] = np.where(df_clean["restimulus"] == 0, reposo_blocks, df_clean["re_repetition_id"])

print(f"    -> df_train shape: {df_train.shape} | df_test shape: {df_test.shape}")

# 3. Ventaneo Aislado (Windowing Last)
def extract_windows_from_df(df, W, S):
    X_list = []
    y_list = []
    groups = df.groupby(['group_id', 'restimulus'])
    block_count = 1
    gesture_names = {0: "Reposo (0)", 1: "Palma (1)", 2: "Puño (2)", 3: "Paz (3)"}
    print(f"\n{'Bloque ID':<10} | {'Clase Gesto':<12} | {'Duración (muestras)':<20} | {'Ventanas Extraídas':<20}")
    print("-" * 70)
    for (g_id, gesture_class), group_df in groups:
        class_name = gesture_names.get(gesture_class, f"Clase {gesture_class}")
        sig_col = "emg_norm" if "emg_norm" in group_df.columns else "filtered"
        signal_block = group_df[sig_col].values
        L = len(signal_block)
        if L < W:
            print(f"Block {block_count:<5} | {class_name:<12} | {L:<20} | 0 (Descartado: L < W)")
            block_count += 1
            continue
        for start in range(0, L - W + 1, S):
            window = signal_block[start:start + W]
            X_list.append(window)
            y_list.append(gesture_class)
        print(f"Block {block_count:<5} | {class_name:<12} | {L:<20} | {len(range(0, L - W + 1, S)):<20}")
        block_count += 1
    return np.array(X_list), np.array(y_list)

print("\n[*] PASO 5: Aplicando ventaneo aislado e independiente a df_train y df_test...")
print("\n=== EXTRACCIÓN DE VENTANAS EN ENTRENAMIENTO (df_train) ===")
X_train_raw, y_train = extract_windows_from_df(df_train, W, S)

print("\n=== EXTRACCIÓN DE VENTANAS EN PRUEBA (df_test) ===")
X_test_raw, y_test = extract_windows_from_df(df_test, W, S)

print(f"\n[+] Ventaneo finalizado.")
print(f"    -> Ventanas de Entrenamiento generadas: {X_train_raw.shape[0]}")
print(f"    -> Ventanas de Prueba generadas:        {X_test_raw.shape[0]}")

[*] PASO 1: Buscando archivos CSV en /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/raw/myotensor_proto
    -> Se encontró el archivo de S07: ['session_20260426_222602.csv']

[*] PASO 2: Cargando archivo session_20260426_222602.csv...
    -> Muestras totales en crudo: 242160
[*] PASO 3: Aplicando filtro de calidad (valid_flag == 1)
    -> Muestras válidas conservadas: 217963 (outliers eliminados: 24197)

[*] PASO 4: Dividiendo el dataset por bloques de forma estratificada...
   -> Dividiendo Gestos Activos por repetición (group_id = re_repetition_id)...
   -> Dividiendo Reposo por bloques contiguos (block_id)...
    -> df_train shape: (174863, 10) | df_test shape: (43100, 10)

[*] PASO 5: Aplicando ventaneo aislado e independiente a df_train y df_test...

=== EXTRACCIÓN DE VENTANAS EN ENTRENAMIENTO (df_train) ===

Bloque ID  | Clase Gesto  | Duración (muestras)  | Ventanas Extraídas  
----------------------------------------------------------------------
Block 1     | Puño (

## 3. Normalización con StandardScaler

In [3]:
from sklearn.preprocessing import StandardScaler
import joblib

print("[*] PASO 6: Convirtiendo e inyectando dimensión de canal único...")
X_train_raw = np.expand_dims(X_train_raw, axis=-1)
X_test_raw = np.expand_dims(X_test_raw, axis=-1)

y_train = y_train.astype(np.int32)
y_test = y_test.astype(np.int32)

print("[*] PASO 7: Normalizando señales con StandardScaler...")
scaler = StandardScaler()
N_train = X_train_raw.shape[0]
N_test = X_test_raw.shape[0]

X_train_flat = X_train_raw.reshape(N_train * W, 1)
X_test_flat  = X_test_raw.reshape(N_test * W, 1)

scaler.fit(X_train_flat)

X_train = scaler.transform(X_train_flat).reshape(N_train, W, 1)
X_test  = scaler.transform(X_test_flat).reshape(N_test, W, 1)

print(f"    -> Conjunto de Entrenamiento (Shape): X_train={X_train.shape}, y_train={y_train.shape}")
print(f"    -> Conjunto de Prueba (Shape):        X_test={X_test.shape}, y_test={y_test.shape}")

[*] PASO 6: Convirtiendo e inyectando dimensión de canal único...
[*] PASO 7: Normalizando señales con StandardScaler...
    -> Conjunto de Entrenamiento (Shape): X_train=(3272, 300, 1), y_train=(3272,)
    -> Conjunto de Prueba (Shape):        X_test=(802, 300, 1), y_test=(802,)


## 4. Codificación One-Hot para Keras / TensorFlow

In [4]:
print("[*] PASO 8: Creando codificación One-Hot...")
num_classes = 4
y_train_onehot = to_categorical(y_train, num_classes=num_classes)
y_test_onehot = to_categorical(y_test, num_classes=num_classes)

print(f"    -> y_train (One-Hot Shape): {y_train_onehot.shape}")
print(f"    -> y_test (One-Hot Shape):  {y_test_onehot.shape}")

[*] PASO 8: Creando codificación One-Hot...
    -> y_train (One-Hot Shape): (3272, 4)
    -> y_test (One-Hot Shape):  (802, 4)


## 5. Exportación de Tensores

In [5]:
print(f"[*] PASO 9: Guardando todos los tensores binarios en {OUTPUT_DIR}...")

np.save(OUTPUT_DIR / 'X_train.npy', X_train)
np.save(OUTPUT_DIR / 'y_train.npy', y_train_onehot)
np.save(OUTPUT_DIR / 'X_test.npy', X_test)
np.save(OUTPUT_DIR / 'y_test.npy', y_test_onehot)

joblib.dump(scaler, OUTPUT_DIR / 'std_scaler.bin')

print("\n[OK] ¡Guardado Completado con Éxito!")

[*] PASO 9: Guardando todos los tensores binarios en /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/processed/myotensor_proto/tensor...

[OK] ¡Guardado Completado con Éxito!
